In [ ]:
import numpy as np
import sys
import os
from autokmc.structure import build_surface, build_nanoparticle
from ase.visualize.x3d import view_x3d
from autokmc.surface import find_surface_atoms
import copy
from collections import Counter
from autokmc.graph import build_graph

In [ ]:
# Ensure the project root is on the path when running from the autokmc/ subdirectory
sys.path.insert(0, os.path.abspath("../../.."))

In [ ]:
### Load the Allegro/NequIP calculator
import torch
from nequip.ase import NequIPCalculator

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_MODEL_FILE = "asehcocuau.nequip.pt2" if _DEVICE == "cuda" else "cpuhcocuau.nequip.pth"
_MODEL_PATH = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", _MODEL_FILE)
print(f"Using device : {_DEVICE}")
print(f"Model file   : {_MODEL_FILE}")

def make_calc():
    """Return a fresh NequIPCalculator instance loaded from the model."""
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device=_DEVICE,
    )

calc = make_calc()
print(f"Calculator : {calc.__class__.__name__}")
print(f"Model      : {_MODEL_PATH}")

In [ ]:
## Build a Cu(111) surface slab
slab = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 1, 1),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True
)

print(f"\nSlab formula : {slab.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab)}")
cell = slab.get_cell()
print(f"Cell (Å)     : a={cell[0,0]:.3f}  b={cell[1,1]:.3f}  c={cell[2,2]:.3f}")

In [ ]:
### Get surface atoms

surface_mask, surface_indices, method = find_surface_atoms(
    slab,
    which="top",
    tag_atoms=True,   # writes slab.arrays["surface"] for extxyz export
)

print(f"Detection method : {method}")
print(f"Surface atoms    : {surface_mask.sum()} / {len(slab)}")
print(f"Surface indices  : {surface_indices}")

In [ ]:
### Visualise surface atoms in X3D
# Surface atoms are shown as Au (gold), bulk atoms remain as Cu
# so the two populations are visually distinct in x3d.
slab_vis = copy.deepcopy(slab)
symbols = np.array(slab_vis.get_chemical_symbols())
symbols[surface_indices] = "Au"
slab_vis.set_chemical_symbols(symbols.tolist())

view_x3d(slab_vis)

In [ ]:
### Build the graph for the slab
graph = build_graph(slab)
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
# Show breakdown by node type
type_counts = Counter(d["type"] for _, d in graph.nodes(data=True))
for t, n in sorted(type_counts.items()):
    print(f"  {t:10s} : {n}")

In [ ]:
### Build the CO reactant from SMILES
# `[C-]#[O+]` is the canonical SMILES for carbon monoxide.  build_reactant
# embeds a 3-D conformer with RDKit ETKDGv3 + MMFF94, then refines it with
# the supplied ASE calculator.  The resulting Reactant carries the optimised
# Atoms, the connectivity graph (every atom tagged surface=2), the gas-phase
# energy reference, and intramolecular orbit / anchor metadata.
from autokmc.reactants import build_reactant

co = build_reactant("[C-]#[O+]", calculator=make_calc())

print(f"Formula        : {co.atoms.get_chemical_formula()}")
print(f"E_gas (eV)     : {co.energy:.4f}")
print(f"Bond length Å  : "
      f"{np.linalg.norm(co.atoms.get_positions()[1] - co.atoms.get_positions()[0]):.3f}")
print(f"unique_nodes   : {co.unique_nodes}")
print(f"anchor_atoms   : {co.anchor_atoms}")
print(f"anchor_orbit   : {co.anchor_orbit}")

In [ ]:
### Enumerate geometrically feasible CO placements on the Cu(111) slab
# This is the multi-atom analogue of `find_sites_for_element`.  For each
# unique single-atom site of C (anchor A), the function looks for raw
# single-atom O sites (atom B) whose optimised position lies within
# `bond_tolerance` of the C–O bond length.  An "O floats" placement
# (B unbonded) is also added per A iso-class.  Results are reduced by
# isomorphism of the n_shells_pair=1 ego-subgraph of (clique_C ∪ clique_O).
from autokmc.find_multisite import find_multisites_for_diatomic

ms_list = find_multisites_for_diatomic(
    graph,
    co,
    bond_tolerance=0.4,   # Å — surface pair distance vs C–O length
    n_shells_pair=1,
    verbose=True,
)

print(f"\n{len(ms_list)} unique CO placements stored in "
      f"graph.graph['multisites']['{co.smiles}']")

In [ ]:
### Inspect the enumerated placements
# For each iso-class print: the bonded pattern, the surface element of
# each clique, the C and O Cartesian positions, and how many equivalent
# raw placements were folded into this class.
LABELS = {1: "top", 2: "bridge", 3: "hollow"}

def _clique_label(graph, clique):
    if clique is None:
        return "—floating—"
    elems = sorted(graph.nodes[i]["element"] for i in clique)
    return f"{LABELS.get(len(clique), f'{len(clique)}-fold')}({'/'.join(elems)})"

for ms in ms_list:
    c_clique, o_clique = ms.atom_cliques
    pC, pO = ms.positions
    print(f"  iso={ms.iso_class:2d}  "
          f"C@{_clique_label(graph, c_clique):14s}  "
          f"O@{_clique_label(graph, o_clique):14s}  "
          f"d(C,O)={np.linalg.norm(pO - pC):.3f} Å  "
          f"members={len(ms.members)}")

In [ ]:
### Visualise one bonded placement on the slab
# Pick the first iso-class where both C and O are bonded to the surface
# and overlay the adsorbate on a copy of the slab (gold = surface atoms).
from ase import Atom

bonded = next(
    (ms for ms in ms_list if all(c is not None for c in ms.atom_cliques)),
    None,
)

In [ ]:
if bonded is None:
    print("No bonded CO placement found.")
else:
    slab_with_co = copy.deepcopy(slab_vis)
    slab_with_co.append(Atom("C", position=bonded.positions[0]))
    slab_with_co.append(Atom("O", position=bonded.positions[1]))
    print(f"Showing iso-class {bonded.iso_class}: "
          f"C@{_clique_label(graph, bonded.atom_cliques[0])}  "
          f"O@{_clique_label(graph, bonded.atom_cliques[1])}")
    view_x3d(slab_with_co)

In [ ]:
### General N-atom enumerator: HCO (formyl) on Cu(111)
# `find_multisites_for_reactant` generalises the diatomic enumerator to
# arbitrary molecules.  Only `reactant.anchor_atoms` (convex-hull-exposed
# atoms — i.e. atoms that aren't sterically buried by the rest of the
# molecule) are eligible to bond to a surface clique.  Every other atom
# floats, with its position reconstructed by rigid (Kabsch) alignment of
# the gas-phase reactant geometry to the chosen surface anchors.
#
# Algorithm:
#   1. Pick a non-empty subset S of anchor atoms (canonicalised by
#      intramolecular orbit so symmetry-equivalent subsets aren't
#      revisited).
#   2. Place the first anchor in S at every unique iso-class site of its
#      element.
#   3. Chain-place each subsequent anchor at any raw single-atom site of
#      its element whose MIC distance to all previously-placed anchors
#      matches the intramolecular distance within `bond_tolerance`.
#   4. Reduce by isomorphism of the union-of-cliques ego-subgraph.
from autokmc.reactants import build_reactant
from autokmc.find_multisite import find_multisites_for_reactant

hco = build_reactant("[CH]=O", calculator=make_calc())

print(f"\nFormula        : {hco.atoms.get_chemical_formula()}")
print(f"E_gas (eV)     : {hco.energy:.4f}")
print(f"unique_nodes   : {hco.unique_nodes}")
print(f"anchor_atoms   : {hco.anchor_atoms}   "
      f"(H typically buried → not an anchor)")
print(f"anchor_orbit   : {hco.anchor_orbit}")

In [ ]:
ms_hco = find_multisites_for_reactant(
    graph,
    hco,
    bond_tolerance=0.4,    # Å — surface pair distance vs intramolecular
    n_shells_pair=1,
    include_partial=True,  # also enumerate physisorbed (singleton anchor)
    verbose=True,
)

print(f"\n{len(ms_hco)} unique HCO placements stored in "
      f"graph.graph['multisites']['{hco.smiles}']")

In [ ]:
### Inspect the enumerated HCO placements
elem_of = {i: hco.graph.nodes[i]["element"] for i in range(len(hco.atoms))}

for ms in ms_hco:
    parts = []
    for i, c in enumerate(ms.atom_cliques):
        parts.append(f"{elem_of[i]}{i}@{_clique_label(graph, c)}")
    bonded_idx = [i for i, c in enumerate(ms.atom_cliques) if c is not None]
    if len(bonded_idx) >= 2:
        i, j = bonded_idx[0], bonded_idx[1]
        d = np.linalg.norm(ms.positions[j] - ms.positions[i])
        d_ref = np.linalg.norm(
            hco.atoms.get_positions()[j] - hco.atoms.get_positions()[i]
        )
        dtag = f"  d({elem_of[i]}{i},{elem_of[j]}{j})={d:.2f}/{d_ref:.2f} Å"
    else:
        dtag = "  (single anchor)"
    print(f"  iso={ms.iso_class:2d}  " + "  ".join(parts) +
          dtag + f"  members={len(ms.members)}")

In [ ]:
### Visualise the first fully-bonded HCO placement
from ase import Atom

full_anchor = next(
    (ms for ms in ms_hco
     if all(ms.atom_cliques[i] is not None for i in hco.anchor_atoms)),
    None,
)

In [ ]:
if full_anchor is None:
    print("No fully-anchored HCO placement found.")
else:
    slab_with_hco = copy.deepcopy(slab_vis)
    for i, p in enumerate(full_anchor.positions):
        slab_with_hco.append(Atom(elem_of[i], position=p))
    print(f"Showing HCO iso-class {full_anchor.iso_class}")
    for i, c in enumerate(full_anchor.atom_cliques):
        print(f"  {elem_of[i]}{i} → {_clique_label(graph, c)}")
    view_x3d(slab_with_hco)